# ML: Delivery Time Prediction

Predicts truck dwell times at stores and distribution centers using
distributed Spark ML gradient-boosted tree regression with MLflow tracking.

## Business Value
- Optimize receiving dock scheduling
- Reduce driver wait times and costs
- Improve labor planning for receiving staff
- Better ETA estimates for inventory availability

## Data Flow
```
Silver (configured fact/dimension tables) --> ML Model --> Gold (configured predictions table)
```

## Model Details
- **Algorithm**: GBTRegressor (Spark ML — distributed, native to Fabric)
- **Target**: Dwell time (minutes) = departed_ts - arrived_ts
- **Features**: Location type, arrival hour, day of week, truck type, historical averages
- **Metrics**: MAE < 15 minutes, MAPE < 25%
- **Output**: Point predictions + configurable empirical prediction intervals based on residual quantiles
- **Tracking**: MLflow experiment with parameters, metrics, and logged model

## Usage
Schedule this notebook at the cadence that fits your SLA (for example daily or more frequently) using the PARAMETERS block defaults or environment overrides.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from datetime import datetime, timezone
import mlflow
import os

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="delivery_prediction")
FACT_TRUCK_MOVES_TABLE = get_env("FACT_TRUCK_MOVES_TABLE", default="fact_truck_moves")
DIM_TRUCKS_TABLE = get_env("DIM_TRUCKS_TABLE", default="dim_trucks")
DIM_STORES_TABLE = get_env("DIM_STORES_TABLE", default="dim_stores")
DIM_DCS_TABLE = get_env("DIM_DCS_TABLE", default="dim_distribution_centers")
OUTPUT_TABLE = get_env("OUTPUT_TABLE", default="dwell_predictions")
MODEL_ARTIFACT_NAME = get_env("MODEL_ARTIFACT_NAME", default="gbt_dwell_model")
RUN_NAME_PREFIX = get_env("MLFLOW_RUN_PREFIX", default="gbt_dwell")

ARRIVED_STATUS = "ARRIVED"
DEPARTED_STATUS_VALUES = ["DEPARTED", "COMPLETED"]
ARRIVAL_TIMESTAMP_CANDIDATES = ["arrival_time", "eta", "event_ts"]
DEPARTURE_TIMESTAMP_CANDIDATES = ["departure_time", "etd", "event_ts"]

# Model parameters
TEST_SIZE = float(get_env("TEST_SIZE", default="0.2"))
TARGET_MAE = float(get_env("TARGET_MAE", default="15.0"))
TARGET_MAPE = float(get_env("TARGET_MAPE", default="0.25"))
MAX_VALID_DWELL_MINUTES = float(get_env("MAX_VALID_DWELL_MINUTES", default="480"))
INTERVAL_COVERAGE = float(get_env("INTERVAL_COVERAGE", default="0.80"))
RANDOM_SEED = int(get_env("RANDOM_SEED", default="42"))

# Spark ML GBT hyperparameters
GBT_MAX_ITER = int(get_env("GBT_MAX_ITER", default="120"))
GBT_MAX_DEPTH = int(get_env("GBT_MAX_DEPTH", default="5"))
GBT_STEP_SIZE = float(get_env("GBT_STEP_SIZE", default="0.05"))
GBT_SUBSAMPLING_RATE = float(get_env("GBT_SUBSAMPLING_RATE", default="0.8"))
GBT_MAX_BINS = int(get_env("GBT_MAX_BINS", default="64"))



if not 0 < TEST_SIZE < 1:
    raise ValueError("TEST_SIZE must be between 0 and 1")
if not 0 < INTERVAL_COVERAGE < 1:
    raise ValueError("INTERVAL_COVERAGE must be between 0 and 1")

LOWER_RESIDUAL_QUANTILE = (1.0 - INTERVAL_COVERAGE) / 2.0
UPPER_RESIDUAL_QUANTILE = 1.0 - LOWER_RESIDUAL_QUANTILE

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {FACT_TRUCK_MOVES_TABLE}, {DIM_TRUCKS_TABLE}, {DIM_STORES_TABLE}, {DIM_DCS_TABLE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Model artifact: {MODEL_ARTIFACT_NAME}")
print(f"Model Targets: MAE < {TARGET_MAE} min, MAPE < {TARGET_MAPE*100}%")
print(f"GBT: {GBT_MAX_ITER} iters, depth={GBT_MAX_DEPTH}, step={GBT_STEP_SIZE}, subsampling={GBT_SUBSAMPLING_RATE}")
print(f"Empirical interval coverage target: {INTERVAL_COVERAGE*100:.1f}%")
print(f"Max valid dwell threshold: {MAX_VALID_DWELL_MINUTES} minutes")
print(f"Departure statuses: {', '.join(DEPARTED_STATUS_VALUES)}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS & MLFLOW SETUP
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")
    print(f"Database '{name}' ready.")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def save_gold(df, table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def pick_existing_column(df, candidate_columns, label):
    for column_name in candidate_columns:
        if column_name in df.columns:
            return column_name
    raise RuntimeError(
        f"Unable to find {label}. Checked columns: {', '.join(candidate_columns)}. "
        f"Available columns: {', '.join(df.columns)}"
    )

def resolve_table_column(df, table_name, *candidates):
    columns_by_lower = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. Available columns: {df.columns}"
    )

def normalize_timestamp(column_name):
    return F.to_timestamp(F.col(column_name).cast("string"))

ensure_database(GOLD_DB)

# Fabric auto-configures the MLflow tracking URI
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

---
## Part 1: Data Preparation

Calculate dwell times from truck arrivals and departures.

In [ ]:
print("=" * 60)
print("PREPARING DWELL TIME DATA")
print("=" * 60)

if not silver_exists(FACT_TRUCK_MOVES_TABLE):
    raise RuntimeError(f"{FACT_TRUCK_MOVES_TABLE} table not found in Silver layer")

# Read fact table — contains ARRIVED and DEPARTED events
df_truck_moves = read_silver(FACT_TRUCK_MOVES_TABLE)

print(f"Total truck move records: {df_truck_moves.count()}")
df_truck_moves.groupBy("status").count().show()

In [ ]:
# Join ARRIVED and departure events by shipment_id to calculate dwell time
arrival_ts_col = pick_existing_column(df_truck_moves, ARRIVAL_TIMESTAMP_CANDIDATES, "arrival timestamp column")
departure_ts_col = pick_existing_column(df_truck_moves, DEPARTURE_TIMESTAMP_CANDIDATES, "departure timestamp column")

print(f"Using arrival timestamp column: {arrival_ts_col}")
print(f"Using departure timestamp column: {departure_ts_col}")

df_arrived = (
    df_truck_moves
    .filter(F.upper(F.col("status")) == ARRIVED_STATUS)
    .select(
        F.col("shipment_id"),
        F.col("truck_id"),
        F.col("store_id"),
        F.col("dc_id"),
        normalize_timestamp(arrival_ts_col).alias("arrived_ts"),
        normalize_timestamp("event_ts").alias("arrived_event_ts"),
    )
    .groupBy("shipment_id")
    .agg(
        F.first("truck_id", ignorenulls=True).alias("truck_id"),
        F.first("store_id", ignorenulls=True).alias("store_id"),
        F.first("dc_id", ignorenulls=True).alias("dc_id"),
        F.min("arrived_ts").alias("arrived_ts"),
        F.min("arrived_event_ts").alias("arrived_event_ts"),
    )
)

df_departed = (
    df_truck_moves
    .filter(F.upper(F.col("status")).isin(DEPARTED_STATUS_VALUES))
    .select(
        F.col("shipment_id"),
        normalize_timestamp(departure_ts_col).alias("departed_ts"),
        normalize_timestamp("event_ts").alias("departed_event_ts"),
    )
    .groupBy("shipment_id")
    .agg(
        F.max("departed_ts").alias("departed_ts"),
        F.max("departed_event_ts").alias("departed_event_ts"),
    )
)

# Join to get complete shipments with both arrival and departure
df_dwell = (
    df_arrived
    .join(df_departed, on="shipment_id", how="inner")
    .withColumn("arrived_ts", normalize_timestamp("arrived_ts"))
    .withColumn("departed_ts", normalize_timestamp("departed_ts"))
    .filter(F.col("arrived_ts").isNotNull() & F.col("departed_ts").isNotNull())
    .withColumn(
        "dwell_minutes",
        (F.unix_timestamp("departed_ts") - F.unix_timestamp("arrived_ts")) / 60,
    )
    .filter(
        (F.col("dwell_minutes") > 0)
        & (F.col("dwell_minutes") < F.lit(MAX_VALID_DWELL_MINUTES))
    )
)

dwell_count = df_dwell.count()
if dwell_count == 0:
    raise RuntimeError("No valid dwell records were produced from ARRIVED and departure events")

print(f"\nShipments with complete dwell times: {dwell_count}")

df_dwell.select(
    F.min("dwell_minutes").alias("min_dwell"),
    F.avg("dwell_minutes").alias("avg_dwell"),
    F.max("dwell_minutes").alias("max_dwell"),
    F.stddev("dwell_minutes").alias("stddev_dwell"),
).show()

---
## Part 2: Feature Engineering

Create temporal, location, and historical features.

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Temporal features from arrival time
df_features = (
    df_dwell
    .withColumn("arrival_hour", F.hour("arrived_ts"))
    .withColumn("arrival_day_of_week", F.dayofweek("arrived_ts"))
    .withColumn(
        "is_weekend",
        F.when(F.col("arrival_day_of_week").isin([1, 7]), 1).otherwise(0),
    )
    .withColumn("arrival_date", F.to_date("arrived_ts"))
)

# Location type feature
df_features = df_features.withColumn(
    "location_type",
    F.when(F.col("store_id").isNotNull(), "STORE").otherwise("DC"),
)

# Get location ID (store_id or dc_id)
df_features = df_features.withColumn(
    "location_id",
    F.when(F.col("store_id").isNotNull(), F.col("store_id")).otherwise(F.col("dc_id")),
)

print(f"Features created: {df_features.count()} records")

In [ ]:
# Join with dimension tables for additional context
print("\nEnriching with dimension data...")

df_trucks_raw = read_silver(DIM_TRUCKS_TABLE)
truck_id_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "truck_id", "id", "ID")
truck_refrigeration_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "refrigeration", "Refrigeration")
truck_home_dc_col = resolve_table_column(df_trucks_raw, DIM_TRUCKS_TABLE, "dc_id", "DCID")
df_trucks = df_trucks_raw.select(
    F.col(truck_id_col).alias("truck_id"),
    F.col(truck_refrigeration_col).cast("double").alias("truck_refrigerated"),
    F.col(truck_home_dc_col).cast("double").alias("truck_home_dc_id"),
)

df_stores_raw = read_silver(DIM_STORES_TABLE)
store_id_col = resolve_table_column(df_stores_raw, DIM_STORES_TABLE, "store_id", "id", "ID")
store_geography_col = resolve_table_column(df_stores_raw, DIM_STORES_TABLE, "geography_id", "GeographyID")
df_stores = df_stores_raw.select(
    F.col(store_id_col).alias("store_id"),
    F.col(store_geography_col).cast("double").alias("store_geography_id"),
)

df_dcs_raw = read_silver(DIM_DCS_TABLE)
dc_id_col = resolve_table_column(df_dcs_raw, DIM_DCS_TABLE, "dc_id", "id", "ID")
dc_geography_col = resolve_table_column(df_dcs_raw, DIM_DCS_TABLE, "geography_id", "GeographyID")
df_dcs = df_dcs_raw.select(
    F.col(dc_id_col).alias("dc_id"),
    F.col(dc_geography_col).cast("double").alias("dc_geography_id"),
)

df_features = (
    df_features
    .join(df_trucks, on="truck_id", how="left")
    .join(df_stores, on="store_id", how="left")
    .join(df_dcs, on="dc_id", how="left")
    .withColumn("truck_refrigerated", F.coalesce(F.col("truck_refrigerated"), F.lit(0.0)))
    .withColumn("truck_home_dc_id", F.coalesce(F.col("truck_home_dc_id"), F.lit(-1.0)))
    .withColumn(
        "location_geography_id",
        F.coalesce(F.col("store_geography_id"), F.col("dc_geography_id"), F.lit(-1.0)),
    )
    .withColumn("location_id", F.col("location_id").cast("double"))
)

print("Dimension enrichment complete.")

In [ ]:
# Build the leakage-safe base feature dataset before train/test split
print("\nPreparing leakage-safe base feature dataset...")

base_numeric_feature_cols = [
    "arrival_hour",
    "arrival_day_of_week",
    "is_weekend",
    "truck_refrigerated",
    "truck_home_dc_id",
    "location_id",
    "location_geography_id",
]
historical_feature_cols = [
    "location_avg_dwell",
    "location_shipment_count",
    "hour_avg_dwell",
]
numeric_feature_cols = base_numeric_feature_cols + historical_feature_cols
target_col = "dwell_minutes"
assembler_input_cols = numeric_feature_cols + ["location_type_idx"]

keep_cols = (
    ["shipment_id", "arrived_ts", "departed_ts", "location_type"]
    + base_numeric_feature_cols
    + [target_col]
)

df_model_base = df_features.select(keep_cols).na.drop().cache()
model_base_count = df_model_base.count()

if model_base_count == 0:
    raise RuntimeError("No records available for modeling after base feature preparation")

print(f"Base feature set ready: {model_base_count} records")

---
## Part 3: Model Training

Train a distributed Spark ML GBT regressor for point prediction and derive
empirical prediction intervals from residual quantiles.

In [ ]:
print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

df_train_base, df_test_base = df_model_base.randomSplit([1.0 - TEST_SIZE, TEST_SIZE], seed=RANDOM_SEED)

train_base_count = df_train_base.count()
test_base_count = df_test_base.count()

if train_base_count == 0:
    raise RuntimeError("Training split is empty; adjust TEST_SIZE or source data volume")
if test_base_count == 0:
    raise RuntimeError("Test split is empty; adjust TEST_SIZE or source data volume")

def build_historical_feature_lookups(df_train_history):
    df_location_avg = (
        df_train_history
        .groupBy("location_id", "location_type")
        .agg(
            F.avg(target_col).alias("location_avg_dwell"),
            F.count("*").cast("double").alias("location_shipment_count"),
        )
    )

    df_hour_avg = (
        df_train_history
        .groupBy("arrival_hour")
        .agg(F.avg(target_col).alias("hour_avg_dwell"))
    )

    global_avg_row = df_train_history.agg(F.avg(target_col).alias("global_avg_dwell")).first()
    global_avg_dwell = float(global_avg_row["global_avg_dwell"]) if global_avg_row and global_avg_row["global_avg_dwell"] is not None else 0.0
    return df_location_avg, df_hour_avg, global_avg_dwell

def attach_historical_features(df_input, df_location_avg, df_hour_avg, global_avg_dwell):
    return (
        df_input
        .join(df_location_avg, on=["location_id", "location_type"], how="left")
        .join(df_hour_avg, on="arrival_hour", how="left")
        .withColumn("location_avg_dwell", F.coalesce(F.col("location_avg_dwell"), F.lit(global_avg_dwell)))
        .withColumn("location_shipment_count", F.coalesce(F.col("location_shipment_count"), F.lit(0.0)))
        .withColumn("hour_avg_dwell", F.coalesce(F.col("hour_avg_dwell"), F.lit(global_avg_dwell)))
    )

print("\nCalculating historical averages from training split only...")
df_location_avg_train, df_hour_avg_train, global_avg_dwell = build_historical_feature_lookups(df_train_base)

df_train = attach_historical_features(df_train_base, df_location_avg_train, df_hour_avg_train, global_avg_dwell).cache()
df_test = attach_historical_features(df_test_base, df_location_avg_train, df_hour_avg_train, global_avg_dwell).cache()
df_model = attach_historical_features(df_model_base, df_location_avg_train, df_hour_avg_train, global_avg_dwell).cache()

train_count = df_train.count()
test_count = df_test.count()
model_count = df_model.count()

if train_count == 0:
    raise RuntimeError("Training dataset is empty after attaching historical features")
if test_count == 0:
    raise RuntimeError("Test dataset is empty after attaching historical features")

print(f"Records for modeling: {model_count}")

indexer = StringIndexer(
    inputCol="location_type",
    outputCol="location_type_idx",
    handleInvalid="keep",
)
assembler = VectorAssembler(
    inputCols=assembler_input_cols,
    outputCol="features",
    handleInvalid="skip",
)
gbt = GBTRegressor(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="predicted_dwell_minutes",
    maxIter=GBT_MAX_ITER,
    maxDepth=GBT_MAX_DEPTH,
    stepSize=GBT_STEP_SIZE,
    subsamplingRate=GBT_SUBSAMPLING_RATE,
    maxBins=GBT_MAX_BINS,
    seed=RANDOM_SEED,
)
point_pipeline = Pipeline(stages=[indexer, assembler, gbt])

print(f"Training set: {train_count} samples")
print(f"Test set: {test_count} samples")
print(f"Features ({len(assembler_input_cols)}): {', '.join(assembler_input_cols)}")

In [ ]:
print("\nTraining distributed Spark ML GBT model...")

lower_residual_offset = None
upper_residual_offset = None

def add_prediction_intervals(df):
    if lower_residual_offset is None or upper_residual_offset is None:
        raise RuntimeError("Prediction intervals are not calibrated yet; run the training cell first")

    return (
        df.withColumn(
            "lower_bound_minutes",
            F.greatest(
                F.lit(0.0),
                F.col("predicted_dwell_minutes") + F.lit(lower_residual_offset),
            ),
        )
        .withColumn(
            "upper_bound_minutes",
            F.greatest(
                F.col("lower_bound_minutes"),
                F.col("predicted_dwell_minutes") + F.lit(upper_residual_offset),
            ),
        )
    )

with mlflow.start_run(
    run_name=f"{RUN_NAME_PREFIX}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "gbt_regressor_sparkml",
        "fact_truck_moves_table": FACT_TRUCK_MOVES_TABLE,
        "dim_trucks_table": DIM_TRUCKS_TABLE,
        "dim_stores_table": DIM_STORES_TABLE,
        "dim_dcs_table": DIM_DCS_TABLE,
        "output_table": OUTPUT_TABLE,
        "max_valid_dwell_minutes": MAX_VALID_DWELL_MINUTES,
        "gbt_max_iter": GBT_MAX_ITER,
        "gbt_max_depth": GBT_MAX_DEPTH,
        "gbt_step_size": GBT_STEP_SIZE,
        "gbt_subsampling_rate": GBT_SUBSAMPLING_RATE,
        "gbt_max_bins": GBT_MAX_BINS,
        "interval_coverage": INTERVAL_COVERAGE,
        "feature_count": len(assembler_input_cols),
        "test_size": TEST_SIZE,
        "train_rows": train_count,
        "test_rows": test_count,
    })

    model_point = point_pipeline.fit(df_train)
    print("  Point prediction model trained.")

    df_train_pred = model_point.transform(df_train).cache()
    residuals_df = df_train_pred.select(
        (F.col(target_col) - F.col("predicted_dwell_minutes")).alias("residual")
    )
    residual_quantiles = residuals_df.approxQuantile(
        "residual",
        [LOWER_RESIDUAL_QUANTILE, UPPER_RESIDUAL_QUANTILE],
        0.01,
    )

    if len(residual_quantiles) != 2:
        raise RuntimeError("Unable to compute empirical residual interval quantiles")

    lower_residual_offset, upper_residual_offset = residual_quantiles
    print(
        f"  Residual interval offsets calibrated at {LOWER_RESIDUAL_QUANTILE:.2f}/{UPPER_RESIDUAL_QUANTILE:.2f}: "
        f"{lower_residual_offset:.2f}, {upper_residual_offset:.2f}"
    )

    df_test_pred = add_prediction_intervals(model_point.transform(df_test)).cache()

    evaluator_mae = RegressionEvaluator(
        labelCol=target_col,
        predictionCol="predicted_dwell_minutes",
        metricName="mae",
    )
    evaluator_rmse = RegressionEvaluator(
        labelCol=target_col,
        predictionCol="predicted_dwell_minutes",
        metricName="rmse",
    )

    mae = evaluator_mae.evaluate(df_test_pred)
    rmse = evaluator_rmse.evaluate(df_test_pred)

    df_eval = df_test_pred.filter(F.col(target_col) > 0)
    mape = df_eval.select(
        F.avg(
            F.abs(F.col(target_col) - F.col("predicted_dwell_minutes"))
            / F.col(target_col)
        )
    ).first()[0] or 0.0

    interval_metrics = df_test_pred.select(
        F.avg(
            F.when(
                (F.col(target_col) >= F.col("lower_bound_minutes"))
                & (F.col(target_col) <= F.col("upper_bound_minutes")),
                F.lit(1.0),
            ).otherwise(F.lit(0.0))
        ).alias("interval_coverage"),
        F.avg(F.col("upper_bound_minutes") - F.col("lower_bound_minutes")).alias("avg_interval_width"),
    ).first()

    actual_interval_coverage = interval_metrics["interval_coverage"] or 0.0
    avg_interval_width = interval_metrics["avg_interval_width"] or 0.0

    mlflow.log_metrics({
        "mae": round(mae, 2),
        "mape": round(mape, 4),
        "rmse": round(rmse, 2),
        "interval_coverage": round(actual_interval_coverage, 4),
        "avg_interval_width": round(avg_interval_width, 2),
    })

    mlflow.spark.log_model(model_point, MODEL_ARTIFACT_NAME)

    print(f"\nMLflow run: {run.info.run_id}")
    print(f"\nTest Set Performance:")
    print(f"  MAE:  {mae:.2f} minutes (Target: < {TARGET_MAE} min)")
    print(f"  MAPE: {mape*100:.2f}% (Target: < {TARGET_MAPE*100}%)")
    print(f"  RMSE: {rmse:.2f} minutes")
    print(f"  Interval Coverage: {actual_interval_coverage*100:.2f}%")
    print(f"  Avg Interval Width: {avg_interval_width:.2f} minutes")

    mae_pass = mae < TARGET_MAE
    mape_pass = mape < TARGET_MAPE

    print(f"\nTarget Achievement:")
    print(f"  MAE Target:  {'PASS' if mae_pass else 'FAIL'}")
    print(f"  MAPE Target: {'PASS' if mape_pass else 'FAIL'}")

    if mae_pass and mape_pass:
        print("\n  Model meets all performance targets!")
    else:
        print("\n  Below target - consider tuning hyperparameters or adding features")

---
## Part 4: Generate Predictions

Create predictions for all completed shipments and save to Gold.

In [ ]:
print("=" * 60)
print("GENERATING PREDICTIONS")
print("=" * 60)

df_all_pred = add_prediction_intervals(model_point.transform(df_model))

df_all_pred = (
    df_all_pred
    .withColumn(
        "prediction_error",
        F.col("dwell_minutes") - F.col("predicted_dwell_minutes"),
    )
    .withColumn(
        "prediction_timestamp",
        F.current_timestamp(),
    )
)

output_cols = [
    "shipment_id",
    "arrived_ts",
    "departed_ts",
    "location_type",
    "arrival_hour",
    "arrival_day_of_week",
    "dwell_minutes",
    "predicted_dwell_minutes",
    "lower_bound_minutes",
    "upper_bound_minutes",
    "prediction_error",
    "prediction_timestamp",
]

df_predictions = df_all_pred.select(output_cols)

print(f"Generated {df_predictions.count()} predictions.")
print("\nSample predictions:")
df_predictions.select(
    "shipment_id",
    "arrived_ts",
    "dwell_minutes",
    "predicted_dwell_minutes",
    "lower_bound_minutes",
    "upper_bound_minutes",
).show(10, truncate=False)

In [ ]:
print("Saving predictions to Gold...")

save_gold(df_predictions, OUTPUT_TABLE)

print(f"\nPredictions saved to {OUTPUT_TABLE}")

---
## Summary

In [ ]:
print("\n" + "=" * 60)
print("DELIVERY TIME PREDICTION COMPLETE")
print("=" * 60)

print(f"\nModel Performance:")
print(f"  MAE:  {mae:.2f} minutes")
print(f"  MAPE: {mape*100:.2f}%")
print(f"  RMSE: {rmse:.2f} minutes")
print(f"  Interval Coverage: {actual_interval_coverage*100:.2f}%")
print(f"  Avg Interval Width: {avg_interval_width:.2f} minutes")

print(f"\nOutput Table:")
print(f"  {LAKEHOUSE_NAME}.{GOLD_DB}.{OUTPUT_TABLE}")
print(f"  Rows: {df_predictions.count()}")

print(f"\nMLflow experiment: {EXPERIMENT_NAME}")
print(f"Model logged: {MODEL_ARTIFACT_NAME}")
print("View results in Fabric > Experiments")

print("\nNext Steps:")
print("  1. Create logistics planning dashboard")
print("  2. Integrate predictions with dock scheduling system")
print("  3. Monitor model performance over time")
print("  4. Retrain on the cadence appropriate for your operations")
print("\nSchedule this notebook with your preferred cadence for fresh predictions.")